# Phân loại bình luận PTIT: Spam và Ham

**Bài tập lớn 1 · Nhóm 1b**  
Nguồn dữ liệu: bình luận trên Facebook group của sinh viên PTIT.  
Mục tiêu: so sánh **BoW / TF-IDF** với **Naive Bayes / SVM** trên cùng một tập kiểm tra.

> **Tình trạng dữ liệu:** Đây là bản thí nghiệm hoàn chỉnh về quy trình, chưa phải bộ lọc spam dùng thực tế. Mẫu spam quá ít; phần kết luận chỉ dựa trên dữ liệu hiện có.

## 1. Nguồn dữ liệu và gán nhãn

Bình luận công khai được thu từ ba group:

1. [Group 408571091380061](https://www.facebook.com/groups/408571091380061/)
2. [Góc Thông Tin PTIT](https://www.facebook.com/groups/584397217391365/)
3. [Cộng đồng sinh viên PTIT](https://www.facebook.com/groups/450310163844833/)

`comments.csv` là dữ liệu thô và tăng thêm khi chạy `crawl_comments.py`; chỉ có cột văn bản, không có tên hoặc ID người dùng. Bản gán nhãn hiện tại `labeled_comments.csv` có **44** dòng dùng cho thí nghiệm. Hai dòng chỉ chứa dấu câu đã bị loại ở lần gán nhãn đầu tiên.

| Nhãn | Quy tắc thực hành |
|---|---|
| `ham` | Hỏi đáp, trao đổi hoặc giao dịch cá nhân có vẻ liên quan đến cộng đồng. |
| `spam` | Mời chào dịch vụ/khóa học, kêu gọi nhắn tin nhằm quảng bá. |

Ba bình luận gán `spam` bắt đầu bằng “Hỗ Trợ Làm ED”, “Dạ bạn thi ở 281”, và “Tui đang gom nhóm học”. **Nhãn cần được nhóm rà soát với ngữ cảnh bài viết gốc.** Không tự tạo thêm mẫu hoặc điều chỉnh nhãn theo dự đoán của mô hình.

In [1]:
import re
import pandas as pd
from IPython.display import display
from underthesea import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

df = pd.read_csv("auto_labeled_comments.csv")
print(f"Số mẫu: {len(df)}")
display(df.head(8))

/home/peterjohn/miniconda3/envs/ai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Số mẫu: 4161


,content,label
0,"k chăm chỉ là đc e, lên là các thầy cho vào ngay",ham
1,Phan Bảo Hai Anh neu m biet m se biet,ham
2,Tôi có,ham
3,giỏi,ham
4,Xứt sắc,ham
5,@nêu bật à tui còn có cả cuốn Kinh tế lượng nữ...,ham
6,Cho pass vs,ham
7,Tadv dễ hơn em ạ,ham


### Phân bố lớp

Kiểm tra số mẫu mỗi lớp trước khi huấn luyện. Mất cân bằng lớp là điểm cần nhớ khi đọc Accuracy.

In [2]:
df["label"].value_counts()

label
ham     3755
spam     406
Name: count, dtype: int64

## 2. Tiền xử lý tiếng Việt

Chuyển chữ thường, bỏ URL và dấu câu; dùng `underthesea` để tách từ. Chỉ bỏ một danh sách stop words ngắn; **giữ từ phủ định** vì chúng có thể đổi ý nghĩa bình luận. Không xóa bình luận ngắn có nội dung.

In [3]:
STOPWORDS = {"là", "thì", "và", "của", "những", "các", "một", "được", "với", "ở", "này", "đó", "cho"}

def preprocess(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text.lower())
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(w for w in word_tokenize(text, format="text").split() if w not in STOPWORDS)

df["processed"] = df["content"].map(preprocess)
display(df[["content", "processed", "label"]].head(8))

,content,processed,label
0,"k chăm chỉ là đc e, lên là các thầy cho vào ngay",k chăm_chỉ đc e lên thầy vào ngay,ham
1,Phan Bảo Hai Anh neu m biet m se biet,phan_bảo hai anh neu m biet m se biet,ham
2,Tôi có,tôi có,ham
3,giỏi,giỏi,ham
4,Xứt sắc,xứt sắc,ham
5,@nêu bật à tui còn có cả cuốn Kinh tế lượng nữ...,nêu bật à tui còn có cả cuốn kinh_tế lượng nữa...,ham
6,Cho pass vs,pass vs,ham
7,Tadv dễ hơn em ạ,tadv dễ hơn em ạ,ham


## 3. Chia dữ liệu

Dùng **80% train / 20% test**, `random_state=42`, chia có giữ tỷ lệ hai lớp. Tất cả vectorizer đều được fit **chỉ trên train** trong `Pipeline`, tránh rò rỉ từ test.

In [4]:
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df["label"]
)
train, test = df.loc[train_idx], df.loc[test_idx]
summary = pd.DataFrame({"Train": train["label"].value_counts(), "Test": test["label"].value_counts()}).fillna(0).astype(int)
display(summary)
print(f"Train: {len(train)} mẫu | Test: {len(test)} mẫu")

,Train,Test
label,,
ham,3003,752
spam,325,81


Train: 3328 mẫu | Test: 833 mẫu


## 4. Huấn luyện bốn tổ hợp

- Đặc trưng: **Bag of Words** (`CountVectorizer`) và **TF-IDF**.
- Thuật toán: **Multinomial Naive Bayes** và **Linear SVM**.
- SVM dùng `class_weight="balanced"` để giảm tác động mất cân bằng lớp. Cùng một tập test được dùng cho cả bốn tổ hợp.

Báo cáo Precision, Recall và F1 của **lớp spam** cùng Accuracy. Khi không dự đoán được spam nào, các chỉ số spam được ghi là 0.

In [5]:
experiments = {
    ("BoW", "Naive Bayes"): (CountVectorizer, MultinomialNB),
    ("BoW", "SVM"): (CountVectorizer, LinearSVC),
    ("TF-IDF", "Naive Bayes"): (TfidfVectorizer, MultinomialNB),
    ("TF-IDF", "SVM"): (TfidfVectorizer, LinearSVC),
}
results, predictions = [], {}
for (feature, algorithm), (vectorizer_cls, model_cls) in experiments.items():
    vectorizer = vectorizer_cls(tokenizer=str.split, token_pattern=None, lowercase=False)
    model = model_cls(class_weight="balanced", random_state=42) if algorithm == "SVM" else model_cls()
    pipeline = make_pipeline(vectorizer, model)
    pipeline.fit(train["processed"], train["label"])
    y_pred = pipeline.predict(test["processed"])
    precision, recall, f1, _ = precision_recall_fscore_support(
        test["label"], y_pred, labels=["spam"], zero_division=0
    )
    results.append({"Đặc trưng": feature, "Mô hình": algorithm,
                    "Accuracy": accuracy_score(test["label"], y_pred),
                    "Precision spam": precision[0], "Recall spam": recall[0], "F1 spam": f1[0]})
    predictions[(feature, algorithm)] = y_pred
scores = pd.DataFrame(results)
display(scores)

,Đặc trưng,Mô hình,Accuracy,Precision spam,Recall spam,F1 spam
0,BoW,Naive Bayes,0.919568,0.659091,0.358025,0.464000
1,BoW,SVM,0.905162,0.513514,0.469136,0.490323
2,TF-IDF,Naive Bayes,0.906363,1.000000,0.037037,0.071429
3,TF-IDF,SVM,0.895558,0.468085,0.543210,0.502857


## 5. Confusion matrix và lỗi dự đoán

Hàng là nhãn thật, cột là nhãn dự đoán. Ô `Spam thật → Dự đoán ham` là spam bị bỏ sót.

In [6]:
for name, y_pred in predictions.items():
    print(name)
    matrix = confusion_matrix(test["label"], y_pred, labels=["ham", "spam"])
    display(pd.DataFrame(matrix, index=["Ham thật", "Spam thật"], columns=["Dự đoán ham", "Dự đoán spam"]))

('BoW', 'Naive Bayes')


,Dự đoán ham,Dự đoán spam
Ham thật,737,15
Spam thật,52,29


('BoW', 'SVM')


,Dự đoán ham,Dự đoán spam
Ham thật,716,36
Spam thật,43,38


('TF-IDF', 'Naive Bayes')


,Dự đoán ham,Dự đoán spam
Ham thật,752,0
Spam thật,78,3


('TF-IDF', 'SVM')


,Dự đoán ham,Dự đoán spam
Ham thật,702,50
Spam thật,37,44


In [7]:
review = test[["content", "label"]].copy().rename(columns={"label": "Nhãn thật"})
for name, y_pred in predictions.items():
    review[f"{name[0]} + {name[1]}"] = y_pred
print("Các mẫu kiểm tra được dự đoán sai bởi ít nhất một mô hình:")
wrong = review[[c for c in review if c not in ("content", "Nhãn thật")]].ne(review["Nhãn thật"], axis=0).any(axis=1)
display(review.loc[wrong])

Các mẫu kiểm tra được dự đoán sai bởi ít nhất một mô hình:


,content,Nhãn thật,BoW + Naive Bayes,BoW + SVM,TF-IDF + Naive Bayes,TF-IDF + SVM
2845,"mình học Toeic nè, học chung k",ham,ham,ham,ham,spam
2050,Bạn nào muốn cc B2 Aptis pass full 10 học phần...,spam,spam,spam,ham,spam
662,cho mình xin tt phòng với,ham,ham,spam,ham,spam
2447,quay tay ít thôi,ham,ham,ham,ham,spam
719,Check ib e với ạ,spam,ham,ham,ham,ham
...,...,...,...,...,...,...
2252,"Cách trg 500m có the toeic lab á, thấy nhìu ng...",ham,spam,spam,ham,spam
2497,Máy giặt dùng chung ở tầng 5.,spam,ham,spam,ham,ham
3378,"Mình hiện đang học bên hiền nhung nè b, mình t...",spam,ham,ham,ham,spam
3251,"Nhưng sốp cíu được, sốp vẫn đnag mở đơn lớp ôn...",spam,ham,ham,ham,spam


## 6. Kết luận

**Kết quả hiện tại:** Bốn tổ hợp cùng đạt Accuracy ≈ **0,889**, nhưng Precision, Recall và F1 của spam đều **0**. Confusion matrix cho thấy cả bốn đều đoán 8 ham đúng và bỏ sót 1 spam trong test. Vì vậy **không thể chọn mô hình tốt nhất** từ kết quả này.

**Giới hạn:** Trong 44 mẫu chỉ có 3 spam; test có đúng 1 spam. Accuracy cao phản ánh việc đoán ham cho hầu hết bình luận, không chứng minh khả năng phát hiện spam.

**Việc cần làm tiếp:** thu thêm bình luận spam thật từ nguồn được phép truy cập; để ít nhất hai người rà soát nhãn với ngữ cảnh bài viết; sau đó chia lại tập test đủ lớn và chạy lại notebook. Giữ nguyên bộ test khi so sánh mô hình.